### **Features recap**

- 52 spectrum features: The measured spectral values at the 52 aligned wavelength channels. So `spectrum_00` is the measured spectrum value at wavelength channel 0, spectrum_01 at channel 1, and so on up to spectrum_51. They describe the shape of that planet's observed spectrum.
- 52 noise features: These correspond in a 1-1 mapping with the 52 spectrum channels. `noise_00` tells us the observational uncertainty/noise associated with `spectrum_00`, `noise_01` corresponds to `spectrum_01`, etc. So the model gets both the measured value and an indication of how reliable that measurement is at each wavelength.
- 7 metadata features: These are broader physical/system properties that do not come from the spectrum itself:
`star_distance`, `star_mass_kg`, `star_radius_m`, `star_temperature`, `planet_mass_kg`, `planet_orbital_period`, and `planet_distance`.

The key point here is that the first to groups of features have an ordered/mapped structure, whereas the 7 metadata variables are standalone.

## **Data Pipeline**

All data is used from the original source files. Nothing is used from the `02_EDA.ipynb` file.

We use only the challenge training data, which is divided into training, validation, calibration and test sets. The Ariel Challenge test set (unseen data) is not used.

In [1]:
from pathlib import Path
import hashlib
import sys
import h5py
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RANDOM_STATE = 42

train_percentage = 0.60
validation_percentage = 0.10
calibration_percentage = 0.15
test_percentage = 0.15

EXPECTED_LABELLED_PLANETS = 41423
EXPECTED_SPECTRAL_CHANNELS = 52
EXPECTED_FEATURE_COUNT = 111

SPECTRUM_FEATURES = [f"spectrum_{channel:02d}" for channel in range(EXPECTED_SPECTRAL_CHANNELS)]

NOISE_FEATURES = [f"noise_{channel:02d}" for channel in range(EXPECTED_SPECTRAL_CHANNELS)]

METADATA_FEATURES = [
    "star_distance",
    "star_mass_kg",
    "star_radius_m",
    "star_temperature",
    "planet_mass_kg",
    "planet_orbital_period",
    "planet_distance"
]

TARGET_COLUMNS = [
    "planet_radius",
    "planet_temp",
    "log_H2O",
    "log_CO2",
    "log_CO",
    "log_CH4",
    "log_NH3"
]

FEATURE_COLUMNS = [*SPECTRUM_FEATURES, *NOISE_FEATURES, *METADATA_FEATURES]

SPECTRAL_PATH = (PROJECT_ROOT / "FullDataset" / "TrainingData" / "SpectralData.hdf5")

AUXILIARY_PATH = (PROJECT_ROOT / "FullDataset" / "TrainingData" / "AuxillaryTable.csv")

TARGET_PATH = (PROJECT_ROOT / "FullDataset" / "TrainingData" / "Ground Truth Package" 
               / "FM_Parameter_Table.csv")

if len(FEATURE_COLUMNS) != EXPECTED_FEATURE_COUNT:
    raise ValueError(f"Expected {EXPECTED_FEATURE_COUNT} features, defined {len(FEATURE_COLUMNS)}")

for required_path in [SPECTRAL_PATH, AUXILIARY_PATH, TARGET_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(f"Required source file not found: {required_path}")


print("Project root:", PROJECT_ROOT)
print("Number of features:", len(FEATURE_COLUMNS))

Project root: /home/ajf23/Documents/Coding new/University stuff/ADC2023/ADC2023-conformal-prediction
Number of features: 111


#### **Load auxiliary metadata and simulated targets**

In [3]:
training_source = pd.read_csv(AUXILIARY_PATH)
target_source = pd.read_csv(TARGET_PATH)

def validation_identifier(table: pd.DataFrame, source_name: str, required_columns: list[str]) -> None:
    """
    Quick data structue checks
    """

    missing_columns = [column for column in required_columns if column not in table.columns]

    if missing_columns:
        raise ValueError(f"{source_name} has missing columns that are required: {missing_columns}")

    if table["planet_ID"].isna().any():
        raise ValueError(f"{source_name} contains missing planet IDs")

    if table["planet_ID"].duplicated().any():
        duplicates = (table.loc[table["planet_ID"].duplicated(keep=False), "planet_ID"].unique().tolist()[:5])
        raise ValueError(f"{source_name} contains duplicate planet IDs: {duplicates}")


validation_identifier(training_source, "Auxiliary metadata", ["planet_ID", *METADATA_FEATURES])
validation_identifier(target_source, "Target table", ["planet_ID", *TARGET_COLUMNS])

labelled_data = (training_source[["planet_ID", *METADATA_FEATURES]]
                 .merge(target_source[["planet_ID", *TARGET_COLUMNS]],
                        on="planet_ID",
                        how="outer",
                        validate="one_to_one",
                        indicator=True
                        )
                    )

unmatched_data = labelled_data["_merge"].ne("both")

if unmatched_data.any():
    unmatched_examples = (labelled_data.loc[unmatched_data, ["planet_ID", "_merge"]
                                            ].head().to_dict(orient="records")
                                            )
    raise ValueError("Metadata and targets do not contain identical planet IDs: {unmatched_examples}")

labelled_data = (labelled_data
                 .drop(columns="_merge")
                 .sort_values("planet_ID", kind="stable")
                 .reset_index(drop=True)
                 )

if len(labelled_data) != EXPECTED_LABELLED_PLANETS:
    raise ValueError(f"Expected {EXPECTED_LABELLED_PLANETS} labelled planets, found {len(labelled_data)}")

print("Auxiliary source shape:", training_source.shape)
print("Target source shape:", target_source.shape)
print("Aligned labelled records:", labelled_data.shape)
print("Unique aligned planet IDs:", labelled_data["planet_ID"].nunique())

Auxiliary source shape: (41423, 9)
Target source shape: (41423, 9)
Aligned labelled records: (41423, 15)
Unique aligned planet IDs: 41423


#### **Load Spectral data**

- 52 spectrum values
- 52 noise values
- 7 metadata variables

In [4]:
planet_ids = labelled_data["planet_ID"].copy()

spectrum_values = np.empty((EXPECTED_LABELLED_PLANETS, EXPECTED_SPECTRAL_CHANNELS), dtype=np.float64)

noise_values = np.empty_like(spectrum_values)

with h5py.File(SPECTRAL_PATH, "r") as spectral_file:
    hdf5_planet_ids = {group_name.removeprefix("Planet_")
                       for group_name in spectral_file.keys()
                       if group_name.startswith("Planet_")
                       }

    expected_planet_ids = set(planet_ids)
    missing_spectral_ids = (expected_planet_ids - hdf5_planet_ids)
    unexpected_spectral_ids = (hdf5_planet_ids - expected_planet_ids)

    if missing_spectral_ids or unexpected_spectral_ids:
        raise ValueError(f"Spectral and tabular planet data differ. Missing spectra: {len(missing_spectral_ids)} "
                         f"Unexpected Spectra: {len(unexpected_spectral_ids)}")

    for row_index, planet_id in enumerate(planet_ids):
        group_name = f"Planet_{planet_id}"
        planet_group = spectral_file[group_name]

        required_datasets = ["instrument_spectrum", "instrument_noise"]
        missing_datasets = [dataset_name for dataset_name in required_datasets
                            if dataset_name not in planet_group]

        if missing_datasets:
            raise ValueError(f"{group_name} is missing data: {missing_datasets}")

        spectrum = np.asarray(planet_group["instrument_spectrum"][()], dtype=np.float64)
        noise = np.asarray(planet_group["instrument_noise"][()], dtype=np.float64)

        if spectrum.shape != (EXPECTED_SPECTRAL_CHANNELS,):
            raise ValueError(f"{group_name}/instrument_spectrum has shape: {spectrum.shape}; expected ({EXPECTED_SPECTRAL_CHANNELS},)")

        if noise.shape != (EXPECTED_SPECTRAL_CHANNELS,):
            raise ValueError(f"{group_name}/instrument_noise has shape {noise.shape}; expected ({EXPECTED_SPECTRAL_CHANNELS},)")

        spectrum_values[row_index] = spectrum
        noise_values[row_index] = noise

print("Spectrum values shape:", spectrum_values.shape)
print("Noise values shape:", noise_values.shape)
print("Missing spectral IDs:", len(missing_spectral_ids))
print("Unexpected spectral IDs:", len(unexpected_spectral_ids))

Spectrum values shape: (41423, 52)
Noise values shape: (41423, 52)
Missing spectral IDs: 0
Unexpected spectral IDs: 0


#### **Construct feature and target matrices**

X follows this block order:

1. `spectrum_00`–`spectrum_51`
2. `noise_00`–`noise_51`;
3. 7 metadata variables

y has the 7 simulated targets.

Both matrices use `planet_ID` as their index to preserve alignment.

In [5]:
metadata_values = labelled_data[METADATA_FEATURES].to_numpy(dtype=np.float64)

# Complete X data/features
# Raw feature values, no preprocessing
raw_features_values = np.column_stack(
    [
        spectrum_values,
        noise_values,
        metadata_values
    ]
)

planet_index = pd.Index(planet_ids.to_numpy(copy=True), name="planet_ID")

X = pd.DataFrame(raw_features_values, index=planet_index, columns=FEATURE_COLUMNS)
y = labelled_data[TARGET_COLUMNS].copy()
y.index = planet_index

all_x_numeric = all(pd.api.types.is_numeric_dtype(X[column]) for column in X.columns)
all_y_numeric = all(pd.api.types.is_numeric_dtype(y[column]) for column in y.columns)

x_missing_count = int(X.isna().sum().sum())
y_missing_count = int(y.isna().sum().sum())

duplicate_planet_ids = int(planet_ids.duplicated().sum())
non_positive_noise = int((noise_values <= 0).sum())

feature_summary = pd.DataFrame(
    {"feature_block":
     ["spectrum",
      "noise",
      "metadata",
      "total"
      ],
      "n_features":
      [len(SPECTRUM_FEATURES),
       len(NOISE_FEATURES),
       len(METADATA_FEATURES),
       len(FEATURE_COLUMNS)
       ]

     }
)

if X.shape != (EXPECTED_LABELLED_PLANETS, EXPECTED_FEATURE_COUNT): # (41423, 111)
    raise ValueError(f"Unexpected X shape: {X.shape}")

if y.shape != (EXPECTED_LABELLED_PLANETS, len(TARGET_COLUMNS)): # (41423, 7)
    raise ValueError(f"Unexpected y shape: {y.shape}")

if not X.index.equals(y.index):
    raise ValueError("X and y planet indices are not aligned")

if not all_x_numeric or not all_y_numeric:
    raise TypeError("Feature or target columns contain non numeric values")

if (x_missing_count or y_missing_count):
    raise ValueError("Features or targets contain missing values.")

if duplicate_planet_ids:
    raise ValueError("Duplicated planet IDs remain after aligment/one-to-one mapping test.")
if non_positive_noise:
    raise ValueError("Noise features contain zero or negative values")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X and y indices aligned:", X.index.equals(y.index))
print("All X columns numeric:", all_x_numeric)
print("All y columns numeric:", all_y_numeric)
print("X missing values:", x_missing_count)
print("y missing values:", y_missing_count)
print("Duplicate planet IDs:", duplicate_planet_ids)
print("Non-positive noise values:", non_positive_noise)

display(feature_summary)

X shape: (41423, 111)
y shape: (41423, 7)
X and y indices aligned: True
All X columns numeric: True
All y columns numeric: True
X missing values: 0
y missing values: 0
Duplicate planet IDs: 0
Non-positive noise values: 0


,feature_block,n_features
0,spectrum,52
1,noise,52
2,metadata,7
3,total,111


### **Splitting the data**

In [14]:
all_planet_ids = planet_ids.to_numpy(copy=True)

ids_train, ids_remaining = train_test_split(all_planet_ids, train_size=train_percentage,
                                            random_state=RANDOM_STATE, shuffle=True)

remaining = 1.0 - train_percentage

val_share = (validation_percentage / remaining)

ids_validation, ids_calibration_and_test = train_test_split(ids_remaining, train_size=val_share,
                                                            random_state=RANDOM_STATE, shuffle=True)

calib_share = (calibration_percentage / (calibration_percentage + test_percentage))

ids_calibration, ids_test = train_test_split(ids_calibration_and_test, train_size=calib_share,
                                            random_state=RANDOM_STATE, shuffle=True)

print("Train IDs:", len(ids_train))
print("Validation IDs:", len(ids_validation))
print("Calibration IDs:", len(ids_calibration))
print("Internal-test IDs:", len(ids_test))

Train IDs: 24853
Validation IDs: 4142
Calibration IDs: 6214
Internal-test IDs: 6214


Put the data splits into new clean variables/DataFrames

In [15]:
X_train = X.loc[ids_train].copy()
X_validation = X.loc[ids_validation].copy()
X_calibration = X.loc[ids_calibration].copy()
X_test = X.loc[ids_test].copy()

y_train = y.loc[ids_train].copy()
y_validation = y.loc[ids_validation].copy()
y_calibration = y.loc[ids_calibration].copy()
y_test = y.loc[ids_test].copy()

### **Verfiy the split's alignment**

Make sure every planet appears exacrly once, each feature row is aligned with its target row.

Should be fine given the verification on my run of code, but best to confirm.

In [19]:
split_ids = {
    "train": ids_train,
    "validation": ids_validation,
    "calibration": ids_calibration,
    "test": ids_test
}

split_objects = {
    "train": (X_train, y_train),
    "validation": (X_validation, y_validation),
    "calibration": (X_calibration, y_calibration),
    "test": (X_test, y_test)
}

split_id_sets = {split_name: set(ids) for split_name, ids in split_ids.items()}

split_names = list(split_ids)

for first_index, first_name in enumerate(split_names):
    for second_name in split_names[first_index + 1:]:
        overlap = (split_id_sets[first_name] & split_id_sets[second_name])

        if overlap:
            raise ValueError(f"{first_name} and {second_name} overlap by {len(overlap)} planet IDs.")

combined_split_ids = np.concatenate(list(split_ids.values()))

if len(combined_split_ids) != EXPECTED_LABELLED_PLANETS:
    raise ValueError("Combined split length does not equal the expected count.")

if len(set(combined_split_ids)) !=EXPECTED_LABELLED_PLANETS:
    raise ValueError("Some planets IDs appear more than once across the splits.")

if set(combined_split_ids) !=set(all_planet_ids):
    raise ValueError("Combined split does not equal the planet set.")

for split_name, (X_split, y_split) in split_objects.items():
    expected_index = pd.Index(split_ids[split_name], name="planet_ID")

    if not X_split.index.equals(expected_index):
        raise ValueError(f"{split_name} feature order differs from its ID order.")

    if not y_split.index.equals(expected_index):
        raise ValueError(f"{split_name} target order differs from its ID order.")

    if not X_split.index.equals(y_split.index):
        raise ValueError(f"{split_name} features and targets are misaligned.")

split_summary = pd.DataFrame(
    {
        "n_planets": {split_name: len(ids) for split_name, ids in split_ids.items()}
    }
)

split_summary["fraction"] = (split_summary["n_planets"] / EXPECTED_LABELLED_PLANETS)

split_summary["requested_percentage"] = [
    train_percentage,
    validation_percentage,
    calibration_percentage,
    test_percentage
]

display(split_summary)
print("Pairswise split overlap: 0")
print("Combined split size:", len(set(combined_split_ids)))
print("All feature and target split indices are still aligned.")

,n_planets,fraction,requested_percentage
train,24853,0.599981,0.60
validation,4142,0.099993,0.10
calibration,6214,0.150013,0.15
test,6214,0.150013,0.15


Pairswise split overlap: 0
Combined split size: 41423
All feature and target split indices are still aligned.


#### **Record the split assignments**
Create a two-column table recording which split contains each `planet_ID`. Later notebooks can use this table to recreate the same training, validation, calibration, and test atasets without making a new split.

In [20]:
split_assignment = pd.concat(
    [
        pd.DataFrame(
            {
                "planet_ID": ids,
                "split": split_name
            }
        )
        for split_name, ids in split_ids.items()
    ], 
    ignore_index=True
)

split_assignment = (split_assignment.sort_values("planet_ID", kind="stable").reset_index(drop=True))

if len(split_assignment) != EXPECTED_LABELLED_PLANETS:
    raise ValueError("Split assignment table has the wrong number of rows")

if not split_assignment["planet_ID"].is_unique:
    raise ValueError("Split assignment table contains duplicate planet IDs.")

if set(split_assignment["split"]) !=set(split_ids):
    raise ValueError("Split assignment table contains unexpected split labels.")


# Converts dataframe into consistent csv text without needing to write a file
assignment_csv_text = split_assignment.to_csv(index=False, lineterminator="\n")

# Creates a reproducibility fingerprint of the exact split assignments.
# Encodes the csv text as bytes, calculates its SHA-256 hash, returns the fingerprint as a 4 char. hexadecimal string.
# If the table is exactly the same in a future run, it will prpduce the same hash.

# If even one planet changes split or the row order changes, the hash will change.
# Essentially checks whether the two tables are identical.
assignment_sha256 = hashlib.sha256(assignment_csv_text.encode("utf-8")).hexdigest()

print("Split assignment shape:", split_assignment.shape)
print("Split assignment SHA-256:", assignment_sha256)
display(split_assignment)

Split assignment shape: (41423, 2)
Split assignment SHA-256: 6b3587361b009d0cd5b0be439d665d3679c3cf4f767ff35ce44496462ec15a74


,planet_ID,split
0,train1,calibration
1,train10,validation
2,train100,train
3,train1000,train
4,train10000,test
...,...,...
41418,train9995,train
41419,train9996,train
41420,train9997,train
41421,train9998,test


**NOTES:**

1. `train1` is the original planet identifier.
    The `train` means it came from the challenge's training package provided for the challenge **NOT** that is must belong to the training split. It correctly is part of the calibration set.

2. The ordering is not by numeric suffix, it is alphanumeric due to the SHA-256 hash.

### **Option to save**

In [21]:
SPLIT_ASSIGNMENT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_assignments_data.csv"
)

# Change to True if you want to save
SAVE_SPLIT_ASSIGNMENTS = False

if SAVE_SPLIT_ASSIGNMENTS:
    SPLIT_ASSIGNMENT_PATH.parent.mkdir(parents=True, exist_ok=True)

    split_assignment.to_csv(SPLIT_ASSIGNMENT_PATH, index=False)
    print("Saved split assignments of data:", SPLIT_ASSIGNMENT_PATH)

else:
    print("Split assignments were not written. Set to True if needed and reviewed.")


Split assignments were not written. Set to True if needed and reviewed.


In [23]:
final_summary = pd.DataFrame(
    {
        "X_shape": {
            "train": X_train.shape,
            "validation": X_validation.shape,
            "calibration": X_calibration.shape,
            "test": X_test.shape
        },
        "y_shape": {
            "train": y_train.shape,
            "validation": y_validation.shape,
            "calibration": y_calibration.shape,
            "test": y_test.shape
        }
    }
)

display(final_summary)
print("Feature columns:", len(FEATURE_COLUMNS))
print("Spectrum features:", len(SPECTRUM_FEATURES))
print("Noise features:", len(NOISE_FEATURES))
print("Metadata features:", len(METADATA_FEATURES))
print("Targets:", len(TARGET_COLUMNS))

,X_shape,y_shape
train,"(24853, 111)","(24853, 7)"
validation,"(4142, 111)","(4142, 7)"
calibration,"(6214, 111)","(6214, 7)"
test,"(6214, 111)","(6214, 7)"


Feature columns: 111
Spectrum features: 52
Noise features: 52
Metadata features: 7
Targets: 7


## **Summary**

- **41,423 planets**
- **111 features**:
  - 52 aligned spectrum channels
  - 52 supplied-noise channels
  - 7 auxiliary metadata variables
- **7 simulator targets** in their original representation

- a reproducible four-way split:
  - 60% training
  - 10% validation
  - 15% calibration
  - 15% test.

The calibration set is reserved for conformal calibration after the
predictive model and its hyperparameters have been fixed. This may not influence
feature selection, preprocessing choices, model fitting, hyperparameter
selection or model selection.

The stage will use only the training and validation sets to
establish predictive baselines and modelling choices.